# 04 — Трекер + top-K keyframes per track


1. На каждом кадре YOLO11n находит ценники, ByteTrack связывает их в треки по `track_id`
2. Для каждого трека выбирает **K_BEST=10 самых резких кадров** (для multi-frame consensus в 06)
3. Считает **capture rate** — сколько GT-ценников попало в треки


**Output:**
- `data/best_crops/<video>/<track_id:04d>_<rank>.jpg` — top-K кадров на трек (rank=0 — самый резкий)
- `data/best_crops_meta.json` — метаданные с полем `crops: [...]`
- `data/per_frame_tracks.json`

In [ ]:
import json
import math
import random
import time
from collections import defaultdict
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch

ROOT = Path.cwd().parent if Path.cwd().name == 'eda' else Path.cwd()
WEIGHTS = ROOT / 'runs' / 'tags_v1' / 'weights' / 'best.pt'   # модель из 03_train_yolo_v2 (Kaggle)
OUT_DIR = ROOT / 'data' / 'best_crops'
META_OUT = ROOT / 'data' / 'best_crops_meta.json'
PER_FRAME_OUT = ROOT / 'data' / 'per_frame_tracks.json'
GT_META = ROOT / 'data' / 'crops_meta.json'

# Авто-выбор устройства: cuda → mps → cpu
if torch.cuda.is_available():
    DEVICE = 'cuda'
elif torch.backends.mps.is_available():
    DEVICE = 'mps'
else:
    DEVICE = 'cpu'

MAX_CROP_SIDE = 600
K_BEST = 10   # top-K кадров на трек (multi-frame consensus в 06)

VIDEOS = [
    ('25_12-20', ROOT / 'Данные/25_12-20/25_12-20.mp4'),
    ('26_12-20', ROOT / 'Данные/26_12-20/26_12-20.mp4'),
    ('43_15', ROOT / 'Данные/43_15/43_15.mp4'),
    ('Unlabeled_25_12-20', ROOT / 'Данные/Unlabeled/25_12-20.mp4'),
    ('Unlabeled_26_12-20', ROOT / 'Данные/Unlabeled/26_12-20.mp4'),
    ('Unlabeled_26_2-10', ROOT / 'Данные/Unlabeled/26_2-10.mp4'),
]

print(f'ROOT:    {ROOT}')
print(f'DEVICE:  {DEVICE}')
print(f'weights: {WEIGHTS}  exists={WEIGHTS.exists()}')
print(f'K_BEST:  {K_BEST}')

## 1. Прогон трекера по всем видео

In [7]:
from ultralytics import YOLO

model = YOLO(str(WEIGHTS))

all_meta = []
per_frame_index = {}
track_counts = {}

t_start = time.time()

for video_id, video_path in VIDEOS:
    if not video_path.exists():
        print(f'WARN: {video_path} не найден — пропуск')
        continue

    print(f'\n[{video_id}] обработка {video_path.name}')

    out_dir = OUT_DIR / video_id

    # Чистая пересборка — структура изменилась на <tid>_<rank>.jpg
    if out_dir.exists():
        for old in out_dir.glob('*.jpg'):
            old.unlink()

    out_dir.mkdir(parents=True, exist_ok=True)

    cap = cv2.VideoCapture(str(video_path))

    fps = cap.get(cv2.CAP_PROP_FPS) or 20.0
    n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    vid_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    vid_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    print(f'  {vid_w}x{vid_h} @ {fps:.1f}fps, {n_frames} кадров')

    results_gen = model.track(
        source=str(video_path),
        tracker='bytetrack.yaml',
        persist=True,
        conf=0.25,
        iou=0.5,
        imgsz=1280,
        stream=True,
        device=DEVICE,
        verbose=False,
    )

    # top_k[tid] = list of (sharp, frame_idx, ts_ms, bbox, crop_resized) — sorted desc, len ≤ K_BEST
    top_k = {}
    frame_idx = 0

    for result in results_gen:
        ok, frame_4k = cap.read()

        if not ok:
            break

        ts_ms = frame_idx * 1000.0 / fps

        boxes = result.boxes

        if boxes is None or boxes.id is None:
            per_frame_index.setdefault(video_id, {})[frame_idx] = []
            frame_idx += 1
            continue

        track_ids = boxes.id.int().tolist()
        bboxes = boxes.xyxy.tolist()

        frame_tracks = []

        for tid, bbox in zip(track_ids, bboxes):
            x1, y1, x2, y2 = bbox

            x1c = max(0, int(x1))
            y1c = max(0, int(y1))
            x2c = min(vid_w, int(x2))
            y2c = min(vid_h, int(y2))

            crop = frame_4k[y1c:y2c, x1c:x2c]

            if crop.size == 0:
                continue

            # Резкость считаем на полном разрешении
            gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
            var = cv2.Laplacian(gray, cv2.CV_64F).var()
            sharp = float(var * math.sqrt(gray.shape[0] * gray.shape[1]))

            frame_tracks.append({
                'track_id': tid,
                'bbox': [x1, y1, x2, y2],
            })

            # Сразу пережимаем — иначе K_BEST копий 4K-кропов съедают RAM
            h, w = crop.shape[:2]
            longest = max(h, w)

            if longest > MAX_CROP_SIDE:
                scale = MAX_CROP_SIDE / longest
                crop_to_keep = cv2.resize(
                    crop,
                    (int(w * scale), int(h * scale)),
                    interpolation=cv2.INTER_AREA,
                )
            else:
                crop_to_keep = crop.copy()

            candidates = top_k.setdefault(tid, [])
            candidates.append((sharp, frame_idx, ts_ms, [x1, y1, x2, y2], crop_to_keep))

            if len(candidates) > K_BEST:
                candidates.sort(key=lambda c: c[0], reverse=True)
                del candidates[K_BEST:]

        per_frame_index.setdefault(video_id, {})[frame_idx] = frame_tracks

        frame_idx += 1

        if frame_idx % 100 == 0:
            print(f'  …{frame_idx}/{n_frames} кадров, треков пока: {len(top_k)}')

    cap.release()

    # Сохраняем top-K кропов на трек
    n_files = 0

    for tid, candidates in sorted(top_k.items()):
        candidates.sort(key=lambda c: c[0], reverse=True)

        crops_meta = []

        for rank, (sharp, fidx, ts, bbox, crop_small) in enumerate(candidates):
            crop_path = out_dir / f'{tid:04d}_{rank}.jpg'

            cv2.imwrite(
                str(crop_path),
                crop_small,
                [cv2.IMWRITE_JPEG_QUALITY, 92],
            )

            crops_meta.append({
                'rank': rank,
                'frame_idx': fidx,
                'timestamp_ms': ts,
                'bbox': bbox,
                'sharpness': round(sharp, 2),
                'crop_path': str(crop_path.relative_to(ROOT)),
            })

            n_files += 1

        # Лучший = rank 0, дублируем поля для обратной совместимости
        best = crops_meta[0]

        all_meta.append({
            'video': video_id,
            'track_id': tid,
            'best_frame_idx': best['frame_idx'],
            'best_timestamp_ms': best['timestamp_ms'],
            'bbox': best['bbox'],
            'frame_wh': [vid_w, vid_h],
            'sharpness': best['sharpness'],
            'crop_path': best['crop_path'],
            'crops': crops_meta,
        })

    track_counts[video_id] = len(top_k)

    print(f'  → {len(top_k)} треков, {n_files} кропов (≤{K_BEST} на трек)')

total_tracks = sum(track_counts.values())
elapsed_min = (time.time() - t_start) / 60

print(f'\nВсего: {total_tracks} треков за {elapsed_min:.1f} мин')


[25_12-20] обработка 25_12-20.mp4
  3840x2160 @ 19.6fps, 823 кадров
  …100/823 кадров, треков пока: 24
  …200/823 кадров, треков пока: 37
  …300/823 кадров, треков пока: 59
  …400/823 кадров, треков пока: 74
  …500/823 кадров, треков пока: 87
  …600/823 кадров, треков пока: 94
  …700/823 кадров, треков пока: 102
  …800/823 кадров, треков пока: 109
  → 109 треков, 909 кропов (≤10 на трек)

[26_12-20] обработка 26_12-20.mp4
  3840x2160 @ 20.0fps, 1776 кадров
  …100/1776 кадров, треков пока: 18
  …200/1776 кадров, треков пока: 29
  …300/1776 кадров, треков пока: 41
  …400/1776 кадров, треков пока: 60
  …500/1776 кадров, треков пока: 67
  …600/1776 кадров, треков пока: 74
  …700/1776 кадров, треков пока: 79
  …800/1776 кадров, треков пока: 81
  …900/1776 кадров, треков пока: 86
  …1000/1776 кадров, треков пока: 100
  …1100/1776 кадров, треков пока: 105
  …1200/1776 кадров, треков пока: 109
  …1300/1776 кадров, треков пока: 118
  …1400/1776 кадров, треков пока: 126
  …1500/1776 кадров, тре

## 2. Сохранение метаданных

In [8]:
META_OUT.write_text(json.dumps(all_meta, ensure_ascii=False, indent=1), encoding='utf-8')

pf_str = {
    vid: {str(fidx): tracks for fidx, tracks in frame_map.items()}
    for vid, frame_map in per_frame_index.items()
}
PER_FRAME_OUT.write_text(json.dumps(pf_str, ensure_ascii=False, indent=1), encoding='utf-8')

print('Saved:')
print(f'  {META_OUT.relative_to(ROOT)}  ({META_OUT.stat().st_size//1024} KB)')
print(f'  {PER_FRAME_OUT.relative_to(ROOT)}  ({PER_FRAME_OUT.stat().st_size//1024} KB)')
print(f'  {OUT_DIR.relative_to(ROOT)}/* — {sum(track_counts.values())} JPEG кропов')

Saved:
  data/best_crops_meta.json  (1521 KB)
  data/per_frame_tracks.json  (6672 KB)
  data/best_crops/* — 544 JPEG кропов


## 3. Capture rate 

Для каждого из 157 GT-ценников ищем track в том же видео, чей bbox пересекается с GT по IoU > 0.5 в нужном кадре (или в окне ±2 кадра — на случай дрожания).

In [9]:
def iou(a, b):
    ax1, ay1, ax2, ay2 = a; bx1, by1, bx2, by2 = b
    ix1 = max(ax1, bx1); iy1 = max(ay1, by1)
    ix2 = min(ax2, bx2); iy2 = min(ay2, by2)
    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    if inter == 0: return 0.0
    return inter / ((ax2 - ax1) * (ay2 - ay1) + (bx2 - bx1) * (by2 - by1) - inter)

gt_records = json.loads(GT_META.read_text())
labeled_videos = {'25_12-20', '26_12-20', '43_15'}
gt_labeled = [g for g in gt_records if g['video'] in labeled_videos]
fps_map = {'25_12-20': 19.6, '26_12-20': 20.0, '43_15': 19.9}

track_lookup = defaultdict(dict)
for p in all_meta:
    track_lookup[p['video']][p['track_id']] = p

matched = 0
per_video = defaultdict(lambda: {'matched': 0, 'total': 0})
frame_dists = []

for gt in gt_labeled:
    vid = gt['video']
    fps = fps_map[vid]
    gt_frame = int(round(gt['frame_timestamp_ms'] / 1000.0 * fps))
    gt_bbox = gt['bbox']
    per_video[vid]['total'] += 1
    frame_map = per_frame_index.get(vid, {})

    best_iou, best_tid = 0.0, None
    for delta in range(-2, 3):
        for t in frame_map.get(gt_frame + delta, []):
            i = iou(gt_bbox, t['bbox'])
            if i > best_iou:
                best_iou, best_tid = i, t['track_id']

    if best_iou >= 0.5 and best_tid is not None:
        matched += 1
        per_video[vid]['matched'] += 1
        td = track_lookup[vid].get(best_tid)
        if td: frame_dists.append(abs(td['best_frame_idx'] - gt_frame))

print(f'CAPTURE RATE: {matched}/{len(gt_labeled)} = {matched/len(gt_labeled):.1%}')
for v in sorted(per_video):
    m, t = per_video[v]['matched'], per_video[v]['total']
    print(f'  {v}: {m}/{t} = {m/t:.1%}')
if frame_dists:
    frame_dists.sort()
    print(f'\nFrame-distance (best vs GT) на {len(frame_dists)} совпавших:')
    print(f'  median={frame_dists[len(frame_dists)//2]}  mean={sum(frame_dists)/len(frame_dists):.1f}  max={max(frame_dists)}')

CAPTURE RATE: 124/157 = 79.0%
  25_12-20: 54/57 = 94.7%
  26_12-20: 42/71 = 59.2%
  43_15: 28/29 = 96.6%

Frame-distance (best vs GT) на 124 совпавших:
  median=29  mean=44.6  max=249
